In [72]:
import re 
import string 
import mlflow 
import nltk
import numpy as np 
import pandas as pd 
import mlflow.sklearn 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from nltk.corpus import stopwords 
from nltk.stem import WordNetLemmatizer 




In [73]:
df =pd.read_csv('IMDB.csv')

In [74]:
df = df.sample(500)
df.to_csv('data.csv',index=False)


In [75]:
df.head()

,review,sentiment
465,This BBC series is excellent. I am no Paleonto...,positive
32,this became a cult movie in chinese college st...,negative
889,"""Journey of Hope"" tells of a poor Turkish fami...",positive
202,Beware My Lovely originated from a play writte...,positive
945,A Disney movie that dares to do something diff...,positive


In [76]:
english_stopwords = set(stopwords.words('english'))
print(english_stopwords)

{'when', "that'll", 'his', "aren't", 'shouldn', 'between', "they'd", 'is', 'than', 'he', 'weren', 'had', 'off', "it'd", 'shan', "isn't", 'only', 'below', 've', 'am', "you've", 'while', 'from', 'until', 'where', 'can', "they're", 'herself', 'mustn', 'during', 'ours', 'through', "we've", 'be', 'd', 'at', 'yours', "he's", "i'd", 'then', 'what', 'she', 'have', 'against', "he'd", 'hers', 'me', 'the', 'why', 'wasn', 'to', 'before', 'that', 'into', "needn't", 'does', 'down', "shan't", 'or', 'which', 'up', 'who', 'ain', 'didn', 'just', 'there', "i'll", 'wouldn', 'because', "you'd", "hasn't", 'for', 'doing', 'itself', 'its', 'how', 'an', 'll', "i'm", 'with', "shouldn't", 'been', 'will', 'whom', 'further', "couldn't", 'their', 'nor', "she'd", 'don', 'by', "mightn't", 'isn', 'other', "she's", "haven't", 'too', 'same', 'once', 'needn', 'ourselves', 'your', "hadn't", 'you', 'himself', 'this', 'above', 'these', "you're", "it's", 't', 'are', 'more', 'over', 'it', 'y', 'yourselves', "mustn't", "wouldn

In [79]:
import nltk
  >>> nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...


True

In [80]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21980\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [81]:
df = normalize_text(df)
df.head()



,review,sentiment
465,bbc series excellent paleontologist series giv...,positive
32,became cult movie chinese college student thou...,negative
889,journey hope tell poor turkish family odyssey ...,positive
202,beware lovely originated play written mel dine...,positive
945,disney movie dare something different least aw...,positive


In [82]:
df['sentiment'].value_counts()

sentiment
positive    250
negative    250
Name: count, dtype: int64

In [83]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]


In [84]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})
df.head()

,review,sentiment
465,bbc series excellent paleontologist series giv...,1
32,became cult movie chinese college student thou...,0
889,journey hope tell poor turkish family odyssey ...,1
202,beware lovely originated play written mel dine...,1
945,disney movie dare something different least aw...,1


In [85]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [86]:
vectorizer=CountVectorizer(max_features=100)
x = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [87]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
